In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## Homogeneous Coordinates

In [ ]:
x1: np.ndarray = np.array([[1, 2], [2, 4]])
x2: np.ndarray = np.array([[1, 2, 1], [2, 4, 2]])

In [ ]:
def to_homogeneous_2d(points: np.ndarray) -> np.ndarray:
    """
    Transforms each point in `points` to homogeneous coordinates.

    :param points: Shape ``(n, 2)``, with ``n`` being the number of points.
    """
    ones: np.ndarray = np.ones((len(points), 1))
    return np.concatenate((points, ones), axis=1)


def to_inhomogeneous_2d(points: np.ndarray) -> np.ndarray:
    """
    Transforms each point in `points` to inhomogeneous coordinates.

    :param points: Shape ``(n, 3)``, with ``n`` being the number of points.
    """
    return (points / points[:, -1][:, None])[:, :2]

In [ ]:
print("Inhomogeneous to homogeneous:", x1, to_homogeneous_2d(x1), sep="\n")

In [ ]:
print("Homogeneous to inhomogeneous:", x2, to_inhomogeneous_2d(x2), sep="\n")

## Points in Lines

In [ ]:
line: np.ndarray = np.array([1, 2, 1]).reshape(-1, 1)
x: np.ndarray = to_homogeneous_2d(np.array([[0, -0.5], [-1, 0], [-1, 0.5]]))

print("Line:\n", line)
print("Points:\n", x)

In [ ]:
def in_line(line: np.ndarray, points: np.ndarray) -> np.ndarray:
    """
    For each point in `points`, determine whether it lies in `line`.

    :param line: Shape ``(3, 1)``.
    :param points: Shape ``(n, 3)``.
    """
    return points @ line == 0.0

In [ ]:
print("Points in lines:", x, in_line(line, x), sep="\n")

## Intersection of Points and Lines

In [ ]:
def intersect(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    Determines the intersection of `a` and `b`.

    :param a: Shape ``(1, 3)``.
    :param b: Shape ``(1, 3)``.
    """
    return np.cross(a, b)

In [ ]:
x1: np.ndarray = np.array([2, 4, 2])
x2: np.ndarray = np.array([0, 1, 1])

line: np.ndarray = intersect(x1, x2)
print("Points", (x1, x2), "intersect on line:", line)
print(x1, "lies in", line, ":", in_line(line, np.array([x1])))
print(x2, "lies in", line, ":", in_line(line, np.array([x2])))

In [ ]:
line1: np.ndarray = np.array([2, 3, 1])
line2: np.ndarray = np.array([0, 1, 2])

x: np.ndarray = intersect(line1, line2)
print("Lines", (line1, line2), "intersect on point:", x)
print(line1, "contains point", x, in_line(line1, np.array([x])))
print(line2, "contains point", x, in_line(line2, np.array([x])))

In [ ]:
line1: np.ndarray = np.array([2, 3, 1])
line2: np.ndarray = np.array([2, 3, 5])

x: np.ndarray = intersect(line1, line2)
print("Lines", (line1, line2), "intersect on ideal point:", x)
print(line1, "contains point", x, in_line(line1, np.array([x])))
print(line2, "contains point", x, in_line(line2, np.array([x])))

## Conic Passing through 5 Points

In [ ]:
points: np.ndarray = np.array(
    [
        [1, 3],
        [1, -3],
        [-1, 3],
        [5, 2],
        [-5, -3],
    ]
)
points = to_homogeneous_2d(points)
print("Points:\n", points)

In [ ]:
def define_conic(points: np.ndarray) -> np.ndarray:
    """
    Constructs a conic that passes through `points`.

    :param points: Shape ``(5, 3)``.
    """
    a: list = []
    for x1, x2, x3 in points:
        a.append((x1 ** 2, x1 * x2, x2 ** 2, x1 * x3, x2 * x3, x3 ** 2))
    a: np.ndarray = np.array(a)
    _, _, sol = np.linalg.svd(a)
    a, b, c, d, e, f = sol[-1]
    return np.array(
        [
            [a, b / 2, d / 2],
            [b / 2, c, e / 2],
            [d / 2, e / 2, f],
        ]
    )

In [ ]:
conic: np.ndarray = define_conic(points)
print("Conic:\n", conic)

In [ ]:
def in_conic(conic: np.ndarray, points: np.ndarray) -> np.ndarray:
    """
    For each point in `points`, determine whether it lies in `conic`.

    :param conic: Shape ``(3, 3)``.
    :param points: Shape ``(n, 3)``.
    """
    return np.isclose(np.diag(points @ conic @ points.T), 0)

In [ ]:
print("Points in conic:", points, in_conic(conic, points), sep="\n")

In [ ]:
def plot_conic(
    C: np.ndarray,
    xlim=(-3, 3),
    ylim=(-3, 3),
    color='r',
    ax=None,
    N=400):
    """
    Plots a conic defined by 3x3 matrix C: [x, y, 1] @ C @ [x, y, 1]^T = 0.

    Generated with ChatGPT.
    """
    x = np.linspace(xlim[0], xlim[1], N)
    y = np.linspace(ylim[0], ylim[1], N)
    X, Y = np.meshgrid(x, y)
    pts = np.stack([X, Y, np.ones_like(X)], axis=-1)
    F = np.einsum('...i,ij,...j->...', pts, C, pts)
    if ax is None:
        fig, ax = plt.subplots()
    ax.contour(X, Y, F, levels=[0], colors=color)
    ax.set_aspect('equal')
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    return ax


plot_conic(conic, xlim=(-10, 10), ylim=(-10, 10))
plt.scatter(points[:, 0], points[:, 1])
plt.show()